# Vocal-Guard — Baseline B0: LFCC-GMM pada ASVspoof 2019 LA
**Fase 0:** Target 13 Oktober 2026 — Official Sanity Check & Single Source of Truth


In [ ]:
!pip install -q soundfile librosa scikit-learn pandas scipy tqdm matplotlib


In [ ]:
import os, sys, glob, math, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.fftpack import dct
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60)
print(f"🔥 PyTorch Device: {device.type.upper()}")
if torch.cuda.is_available():
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
print("=" * 60)


In [ ]:
# Deteksi lokasi protokol secara fleksibel
kaggle_protocols = list(Path("/kaggle/input").glob("**/ASVspoof2019.LA.cm.train.trn.txt"))
if not kaggle_protocols:
    raise FileNotFoundError("Protokol ASVspoof2019 LA tidak ditemukan di /kaggle/input!")

PROTOCOL_DIR = kaggle_protocols[0].parent
LA_ROOT = PROTOCOL_DIR.parent

# Cari folder audio train dan dev
def find_flac_dir(root_dir, split_name):
    candidates = list(root_dir.glob(f"**/*{split_name}*/**/flac"))
    if not candidates:
        candidates = list(Path("/kaggle/input").glob(f"**/*{split_name}*/**/flac"))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f"Folder FLAC untuk split {split_name} tidak ditemukan!")

train_audio_dir = find_flac_dir(LA_ROOT, "train")
dev_audio_dir = find_flac_dir(LA_ROOT, "dev")

def load_protocol(protocol_file):
    df = pd.read_csv(
        protocol_file,
        sep=r"\s+",
        header=None,
        names=["speaker_id", "audio_id", "dash", "system_id", "label"],
        dtype=str
    )
    df["label"] = df["label"].str.lower()
    return df

train_df = load_protocol(PROTOCOL_DIR / "ASVspoof2019.LA.cm.train.trn.txt")
dev_df = load_protocol(PROTOCOL_DIR / "ASVspoof2019.LA.cm.dev.trl.txt")

print("=" * 60)
print(f"✅ Dataset & Protokol Ditemukan:")
print(f"   Train samples: {len(train_df):,} (Bonafide: {(train_df["label"] == "bonafide").sum():,}, Spoof: {(train_df["label"] == "spoof").sum():,})")
print(f"   Dev samples:   {len(dev_df):,} (Bonafide: {(dev_df["label"] == "bonafide").sum():,}, Spoof: {(dev_df["label"] == "spoof").sum():,})")
print(f"   Train FLAC:    {train_audio_dir}")
print(f"   Dev FLAC:      {dev_audio_dir}")
print("=" * 60)


In [ ]:
def get_linear_filterbank(n_filters=20, n_fft=512, sr=16000, low_freq=0.0, high_freq=None):
    if high_freq is None:
        high_freq = sr / 2.0
    freq_points = np.linspace(low_freq, high_freq, n_filters + 2)
    bin_points = np.floor((n_fft + 1) * freq_points / sr).astype(int)
    n_bins = n_fft // 2 + 1
    filterbank = np.zeros((n_filters, n_bins))
    for i in range(n_filters):
        left, center, right = bin_points[i], bin_points[i + 1], bin_points[i + 2]
        if center > left:
            for b in range(left, center):
                if b < n_bins:
                    filterbank[i, b] = (b - left) / (center - left)
        if right > center:
            for b in range(center, right):
                if b < n_bins:
                    filterbank[i, b] = (right - b) / (right - center)
    return filterbank

def compute_deltas(feats, N=2):
    T, D = feats.shape
    deltas = np.zeros_like(feats)
    denom = 2 * sum(n**2 for n in range(1, N + 1))
    for t in range(T):
        num = np.zeros(D)
        for n in range(1, N + 1):
            t_plus = min(T - 1, t + n)
            t_minus = max(0, t - n)
            num += n * (feats[t_plus] - feats[t_minus])
        deltas[t] = num / denom
    return deltas

def extract_lfcc(y, sr=16000, window_ms=20.0, hop_ms=10.0, n_fft=512, num_filters=20, num_coeffs=20):
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    y = np.append(y[0], y[1:] - 0.97 * y[:-1])
    frame_len = int(sr * window_ms / 1000.0)
    hop_len = int(sr * hop_ms / 1000.0)
    if len(y) < frame_len:
        y = np.pad(y, (0, frame_len - len(y)))
    num_frames = 1 + (len(y) - frame_len) // hop_len
    shape = (num_frames, frame_len)
    strides = (y.strides[0] * hop_len, y.strides[0])
    frames = np.lib.stride_tricks.as_strided(y, shape=shape, strides=strides).copy()
    frames *= np.hanning(frame_len)
    mag_spec = np.abs(np.fft.rfft(frames, n=n_fft, axis=-1))
    fbank = get_linear_filterbank(n_filters=num_filters, n_fft=n_fft, sr=sr)
    filter_energies = np.dot(mag_spec, fbank.T)
    log_energies = np.log(filter_energies + 1e-12)
    static_lfcc = dct(log_energies, type=2, axis=-1, norm="ortho")[:, :num_coeffs]
    delta = compute_deltas(static_lfcc, N=2)
    delta2 = compute_deltas(delta, N=2)
    return np.hstack([static_lfcc, delta, delta2])


In [ ]:
def collect_frames(df, audio_dir, target_label, max_frames=200000):
    subset = df[df['label'] == target_label]
    all_frames = []
    print('Mengumpulkan frame training untuk ' + target_label.upper() + '...')
    for _, row in tqdm(subset.iterrows(), total=len(subset)):
        file_path = audio_dir / (str(row['audio_id']) + '.flac')
        if not file_path.exists():
            continue
        y, sr = sf.read(str(file_path), dtype='float32')
        feats = extract_lfcc(y, sr=sr)
        all_frames.append(feats)
        if sum(len(f) for f in all_frames) >= max_frames:
            break
    stacked = np.vstack(all_frames)
    if len(stacked) > max_frames:
        idx = np.random.choice(len(stacked), size=max_frames, replace=False)
        stacked = stacked[idx]
    print('Total frame ' + target_label + ': ' + str(stacked.shape))
    return stacked

MAX_FRAMES = 200000
frames_bonafide = collect_frames(train_df, train_audio_dir, 'bonafide', max_frames=MAX_FRAMES)
frames_spoof = collect_frames(train_df, train_audio_dir, 'spoof', max_frames=MAX_FRAMES)

N_COMPONENTS = 256
print('[1/2] Melatih GMM Bonafide (' + str(N_COMPONENTS) + ' komponen)...')
gmm_bonafide = GaussianMixture(n_components=N_COMPONENTS, covariance_type='diag', max_iter=60, random_state=42)
gmm_bonafide.fit(frames_bonafide)
print('✅ GMM Bonafide selesai!')

print('[2/2] Melatih GMM Spoof (' + str(N_COMPONENTS) + ' komponen)...')
gmm_spoof = GaussianMixture(n_components=N_COMPONENTS, covariance_type='diag', max_iter=60, random_state=42)
gmm_spoof.fit(frames_spoof)
print('✅ GMM Spoof selesai!')

with open('gmm_bonafide.pkl', 'wb') as f:
    pickle.dump(gmm_bonafide, f)
with open('gmm_spoof.pkl', 'wb') as f:
    pickle.dump(gmm_spoof, f)
print('💾 Checkpoint model GMM disimpan!')


In [ ]:
class TorchGMM(torch.nn.Module):
    def __init__(self, sklearn_gmm, device='cuda'):
        super().__init__()
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.weights = torch.tensor(sklearn_gmm.weights_, dtype=torch.float32, device=self.device)
        self.means = torch.tensor(sklearn_gmm.means_, dtype=torch.float32, device=self.device)
        self.covars = torch.tensor(sklearn_gmm.covariances_, dtype=torch.float32, device=self.device)
        self.log_weights = torch.log(self.weights)
        self.log_det = torch.log(self.covars).sum(dim=-1)
        self.D = self.means.shape[-1]
        self.const = self.D * math.log(2.0 * math.pi)

    def score_utterance(self, X_t):
        diff = X_t.unsqueeze(1) - self.means.unsqueeze(0)
        quad = torch.sum((diff ** 2) / self.covars.unsqueeze(0), dim=-1)
        log_p_k = -0.5 * (self.const + self.log_det.unsqueeze(0) + quad) + self.log_weights.unsqueeze(0)
        return torch.logsumexp(log_p_k, dim=-1).mean()

torch_bona = TorchGMM(gmm_bonafide, device=device)
torch_spoof = TorchGMM(gmm_spoof, device=device)

def compute_eer(bonafide_scores, spoof_scores):
    y_true = np.concatenate([np.ones_like(bonafide_scores), np.zeros_like(spoof_scores)])
    scores = np.concatenate([bonafide_scores, spoof_scores])
    fpr, tpr, thresholds = roc_curve(y_true, scores, pos_label=1)
    fnr = 1.0 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return float(eer), float(thresholds[idx])

print('🚀 Menghitung skor evaluasi 24,844 utterance di Dev Set dengan akselerasi GPU...')

def process_row(row):
    file_path = dev_audio_dir / (str(row['audio_id']) + '.flac')
    if not file_path.exists():
        return None
    try:
        y, sr = sf.read(str(file_path), dtype='float32')
        feats = extract_lfcc(y, sr=sr)
        with torch.no_grad():
            x_t = torch.tensor(feats, dtype=torch.float32, device=device)
            score = (torch_bona.score_utterance(x_t) - torch_spoof.score_utterance(x_t)).item()
        return score, row['label']
    except Exception:
        return None

rows = [row for _, row in dev_df.iterrows()]
results = []

with ThreadPoolExecutor(max_workers=8) as ex:
    for r in tqdm(ex.map(process_row, rows), total=len(rows), desc='GPU Dev Scoring'):
        if r is not None:
            results.append(r)

dev_scores = np.array([r[0] for r in results])
dev_labels = np.array([r[1] for r in results])

bona_scores = dev_scores[dev_labels == 'bonafide']
spoof_scores = dev_scores[dev_labels == 'spoof']

eer_dev, threshold = compute_eer(bona_scores, spoof_scores)

print('=' * 60)
print('🎯 HASIL RESMI BASELINE B0 (LFCC-GMM pada ASVspoof 2019 LA Dev):')
print('   • EER Dev:       {:.2f}% ⭐'.format(eer_dev * 100))
print('   • Threshold:     {:.4f}'.format(threshold))
print('   • Bonafide Mean: {:.4f}'.format(bona_scores.mean()))
print('   • Spoof Mean:    {:.4f}'.format(spoof_scores.mean()))
print('=' * 60)

# Simpan log evaluasi
log_df = pd.DataFrame([{
    'Model': 'B0 LFCC-GMM',
    'Fitur': 'LFCC (60-dim: 20 static + delta + delta2)',
    'Window': '4 s / full',
    'Trimming': 'tidak',
    'Seed': 42,
    'Components': N_COMPONENTS,
    'EER dev': '{:.2f}%'.format(eer_dev * 100)
}])
log_df.to_csv('b0_evaluation_log.csv', index=False)
print('✅ Log disimpan ke b0_evaluation_log.csv')
